# Notebook 6: Administración Unity Catalog

## OceanWatch Analytics

### Presentado por: Gupo 2

#### Pablo Daniel Díaz Mera
#### Ana Catalina Gelvez
#### Christian Rene Mendez

## 1. Inventario de Unity Catalog

In [0]:
# Configuración inicial
catalogo = "workspace"
esquema = "oceanwatch"

print("INVENTARIO DE UNITY CATALOG")
print("=" * 60)

# 1. Catálogos disponibles
print("\n1. CATÁLOGOS DISPONIBLES")
display(spark.sql("SHOW CATALOGS"))

# 2. Esquemas del catálogo seleccionado
print(f"\n2. ESQUEMAS DEL CATÁLOGO: {catalogo}")
display(spark.sql(
    f"SHOW SCHEMAS IN `{catalogo}`"
))

# 3. Volúmenes del esquema seleccionado
print(f"\n3. VOLÚMENES DEL ESQUEMA: {catalogo}.{esquema}")
display(spark.sql(
    f"SHOW VOLUMES IN `{catalogo}`.`{esquema}`"
))

# 4. Tablas registradas en el esquema
print(f"\n4. TABLAS DEL ESQUEMA: {catalogo}.{esquema}")
display(spark.sql(
    f"SHOW TABLES IN `{catalogo}`.`{esquema}`"
))


INVENTARIO DE UNITY CATALOG

1. CATÁLOGOS DISPONIBLES


catalog
dbacademy
main
samples
system
workspace



2. ESQUEMAS DEL CATÁLOGO: workspace


databaseName
default
information_schema
oceanwatch



3. VOLÚMENES DEL ESQUEMA: workspace.oceanwatch


database,volume_name
oceanwatch,ais_data
oceanwatch,geo_data



4. TABLAS DEL ESQUEMA: workspace.oceanwatch


database,tableName,isTemporary


## 2. Creación de espacios de trabajo

### Creación de Volumen landing

In [0]:
catalogo = "workspace"
esquema = "oceanwatch"
volumen = "ais_landing"

nombre_completo = f"`{catalogo}`.`{esquema}`.`{volumen}`"

# Crear el volumen si todavía no existe
spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {nombre_completo}
    COMMENT 'Volumen de recepción de archivos AIS para OceanWatch Analytics'
""")

print("VERIFICACIÓN DEL VOLUMEN")
print("=" * 60)

# Consultar los volúmenes registrados
display(spark.sql(
    f"SHOW VOLUMES IN `{catalogo}`.`{esquema}`"
))

# Mostrar la ruta del nuevo volumen
ruta_landing = f"/Volumes/{catalogo}/{esquema}/{volumen}/"

print(f"\nRuta del volumen landing: {ruta_landing}")


VERIFICACIÓN DEL VOLUMEN


database,volume_name
oceanwatch,ais_data
oceanwatch,ais_landing
oceanwatch,geo_data



Ruta del volumen landing: /Volumes/workspace/oceanwatch/ais_landing/


### Creación de directorios

In [0]:
# Configuración
catalogo = "workspace"
esquema = "oceanwatch"

ruta_base = f"/Volumes/{catalogo}/{esquema}"

# Directorios requeridos
directorios = {
    "Landing - CSV entrantes": f"{ruta_base}/ais_landing/incoming",
    "Checkpoints - Streaming": f"{ruta_base}/ais_data/checkpoints",
    "Esquemas - Auto Loader": f"{ruta_base}/ais_data/schemas"
}

print("CREACIÓN DE DIRECTORIOS")
print("=" * 60)

# Crear directorios si no existen
for descripcion, ruta in directorios.items():
    dbutils.fs.mkdirs(ruta)
    print(f"[OK] {descripcion}")
    print(f"     {ruta}")

# Verificar la estructura
print("\nVERIFICACIÓN DE DIRECTORIOS")
print("=" * 60)

for volumen in ["ais_landing", "ais_data"]:
    ruta_volumen = f"{ruta_base}/{volumen}/"

    print(f"\nVOLUMEN: {volumen}")

    for elemento in dbutils.fs.ls(ruta_volumen):
        if elemento.isDir():
            print(f"  [DIR] {elemento.name}")


CREACIÓN DE DIRECTORIOS
[OK] Landing - CSV entrantes
     /Volumes/workspace/oceanwatch/ais_landing/incoming
[OK] Checkpoints - Streaming
     /Volumes/workspace/oceanwatch/ais_data/checkpoints
[OK] Esquemas - Auto Loader
     /Volumes/workspace/oceanwatch/ais_data/schemas

VERIFICACIÓN DE DIRECTORIOS

VOLUMEN: ais_landing
  [DIR] incoming/

VOLUMEN: ais_data
  [DIR] almacenamiento/
  [DIR] checkpoints/
  [DIR] csv/
  [DIR] schemas/
  [DIR] zip/


## 3. Inventario de permisos y propietarios

### Verificación de permisos

In [0]:
catalogo = "workspace"
esquema = "oceanwatch"
volumen = "ais_landing"

print("INVENTARIO DE PERMISOS")
print("=" * 60)

# 1. Usuario que ejecuta el notebook
print("\n1. USUARIO ACTUAL")
display(spark.sql("SELECT current_user() AS usuario_actual"))

# 2. Privilegios sobre el catálogo
print(f"\n2. PRIVILEGIOS DEL CATÁLOGO: {catalogo}")
display(spark.sql(
    f"SHOW GRANTS ON CATALOG `{catalogo}`"
))

# 3. Privilegios sobre el esquema
print(f"\n3. PRIVILEGIOS DEL ESQUEMA: {catalogo}.{esquema}")
display(spark.sql(
    f"SHOW GRANTS ON SCHEMA `{catalogo}`.`{esquema}`"
))

# 4. Privilegios sobre el volumen landing
print(f"\n4. PRIVILEGIOS DEL VOLUMEN: {volumen}")
display(spark.sql(
    f"SHOW GRANTS ON VOLUME `{catalogo}`.`{esquema}`.`{volumen}`"
))


INVENTARIO DE PERMISOS

1. USUARIO ACTUAL


usuario_actual
pd.diaz@uniandes.edu.co



2. PRIVILEGIOS DEL CATÁLOGO: workspace


Principal,ActionType,ObjectType,ObjectKey
_workspace_users_workspace_7474650215306821,USE CATALOG,CATALOG,workspace



3. PRIVILEGIOS DEL ESQUEMA: workspace.oceanwatch


Principal,ActionType,ObjectType,ObjectKey



4. PRIVILEGIOS DEL VOLUMEN: ais_landing


Principal,ActionType,ObjectType,ObjectKey


### Verificación de propietarios

In [0]:
catalogo = "workspace"
esquema = "oceanwatch"

print("VERIFICACIÓN DE PROPIETARIOS")
print("=" * 60)

# 1. Información del esquema
print("\n1. ESQUEMA")
display(spark.sql(
    f"DESCRIBE SCHEMA EXTENDED `{catalogo}`.`{esquema}`"
))

# 2. Información de los volúmenes
for volumen in ["ais_landing", "ais_data", "geo_data"]:

    print(f"\n2. VOLUMEN: {volumen}")

    display(spark.sql(
        f"DESCRIBE VOLUME `{catalogo}`.`{esquema}`.`{volumen}`"
    ))


VERIFICACIÓN DE PROPIETARIOS

1. ESQUEMA


database_description_item,database_description_value
Catalog Name,workspace
Namespace Name,oceanwatch
Comment,
Location,
Owner,pd.diaz@uniandes.edu.co
Properties,"((unity.catalog.managed.iceberg.defaults.delta.feature.catalogManaged,supported))"
Predictive Optimization,ENABLE (inherited from METASTORE metastore_aws_us_east_2)



2. VOLUMEN: ais_landing


name,catalog,database,owner,storage_location,volume_type,comment,securable_type,securable_kind
ais_landing,workspace,oceanwatch,pd.diaz@uniandes.edu.co,,MANAGED,Volumen de recepción de archivos AIS para OceanWatch Analytics,VOLUME,VOLUME_DB_STORAGE



2. VOLUMEN: ais_data


name,catalog,database,owner,storage_location,volume_type,comment,securable_type,securable_kind
ais_data,workspace,oceanwatch,pd.diaz@uniandes.edu.co,,MANAGED,null,VOLUME,VOLUME_DB_STORAGE



2. VOLUMEN: geo_data


name,catalog,database,owner,storage_location,volume_type,comment,securable_type,securable_kind
geo_data,workspace,oceanwatch,pd.diaz@uniandes.edu.co,,MANAGED,null,VOLUME,VOLUME_DB_STORAGE


## 4. Validación de estructura de almacenamiento

In [0]:

catalogo = "workspace"
esquema = "oceanwatch"

rutas = {
    "Landing": f"/Volumes/{catalogo}/{esquema}/ais_landing/incoming/",
    "Checkpoints": f"/Volumes/{catalogo}/{esquema}/ais_data/checkpoints/",
    "Schemas": f"/Volumes/{catalogo}/{esquema}/ais_data/schemas/"
}

print("VALIDACIÓN DE ALMACENAMIENTO")
print("=" * 60)

for nombre, ruta in rutas.items():

    print(f"\n{nombre.upper()}")
    print(f"Ruta: {ruta}")

    try:
        archivos = dbutils.fs.ls(ruta)

        print("[OK] Directorio accesible")
        print(f"Elementos encontrados: {len(archivos)}")

        for archivo in archivos:
            tipo = "DIR" if archivo.isDir() else "FILE"
            print(f"  [{tipo}] {archivo.name}")

    except Exception as e:
        print(f"[ERROR] {type(e).__name__}: {e}")


VALIDACIÓN DE ALMACENAMIENTO

LANDING
Ruta: /Volumes/workspace/oceanwatch/ais_landing/incoming/
[OK] Directorio accesible
Elementos encontrados: 0

CHECKPOINTS
Ruta: /Volumes/workspace/oceanwatch/ais_data/checkpoints/
[OK] Directorio accesible
Elementos encontrados: 0

SCHEMAS
Ruta: /Volumes/workspace/oceanwatch/ais_data/schemas/
[OK] Directorio accesible
Elementos encontrados: 0


## 5. Administración de permisos para descarga de nuevos datos